# Learning curves (labdata)

Reads selected upstream `DecisionTask.TrialSet` rows directly. CLI alternative:

```bash
uv run python scripts/analyses/plot_learning_curves.py --analysis-set-id <id> --output figures/learning.pdf
```

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "behavioral_metrics" else Path.cwd()
for path in [REPO_ROOT, REPO_ROOT / "src"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from labdata.schema import DecisionTask
from labdata_plugin.analysisschema import BehaviorAnalysisSet

ANALYSIS_SET_ID = "example_analysis_set"  # replace after seeding
selected = BehaviorAnalysisSet.TrialSet() & {"analysis_set_id": ANALYSIS_SET_ID}
rows = (DecisionTask.TrialSet() & selected).fetch(as_dict=True)
assert rows, f"No selected TrialSets for {ANALYSIS_SET_ID}"

data = pd.DataFrame(rows).sort_values(["subject_name", "session_name"])
fig, ax = plt.subplots(figsize=(8, 4))
for subject, subject_df in data.groupby("subject_name"):
    ax.plot(subject_df["performance_easy"].to_numpy(), marker="o", label=subject)
ax.set_xlabel("Session index")
ax.set_ylabel("Easy performance")
ax.set_ylim(0, 1)
ax.legend(frameon=False, fontsize=8)
ax.set_title("Easy performance across selected sessions")
fig.show()